# Data Validation

Confirms the three prompt-injection datasets load correctly and have the structure assumed in the project plan.

Use the Python environment `capstone`.

## Structure
- Setup (cells below): environment check and imports. Run every session.
- Section 1: download datasets from HuggingFace to local `data/`. Run once per machine.
- Section 2: EDA from the local snapshots. Run every session.


In [1]:
import sys
print(sys.executable)  # should show a path containing "capstone"
import matplotlib
print(matplotlib.__version__)  # should print cleanly with no warnings

c:\Users\boga\miniconda3\envs\capstone\python.exe
3.10.8


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
from dotenv import load_dotenv
from pathlib import Path

# resolve repo root by walking up from CWD until we find .git; robust to being run from either repo root or notebooks/
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / ".git").exists(), f"Could not locate repo root (.git) starting from {Path.cwd()}"

# loads HF_TOKEN (and any other secrets) from .env at repo root
load_dotenv(dotenv_path=REPO_ROOT / ".env")
assert os.getenv("HF_TOKEN"), "HF_TOKEN not found. Create .env at repo root with HF_TOKEN=hf_..."

# standard paths relative to repo root
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# display settings
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

## Section 1: Download datasets (idempotent)

Downloads the three datasets from HuggingFace and writes them to `data/` (gitignored).

This cell is safe to re-run. It checks for each dataset folder and skips any that already exist, so "Restart Kernel and Run All" will not re-hit HuggingFace after the first successful download.

To force a fresh re-download of a dataset (for example, after an upstream update), delete the corresponding folder (`data/deepset`, `data/neuralchemy`, or `data/spml`) and re-run the cell.


In [ ]:
DATA_DIR.mkdir(exist_ok=True)

# Idempotent: skips any dataset whose local snapshot already exists.
# Delete data/<name> to force a fresh re-download of that dataset.

if not (DATA_DIR / "deepset").exists():
    print("Loading deepset...")
    ds_deepset = load_dataset("deepset/prompt-injections")
    ds_deepset.save_to_disk(DATA_DIR / "deepset")
    print(f"deepset saved: {len(ds_deepset['train'])} train, {len(ds_deepset['test'])} test rows")
else:
    print("deepset: already on disk, skipping.")

if not (DATA_DIR / "neuralchemy").exists():
    print("Loading neuralchemy...")
    ds_neuro = load_dataset("neuralchemy/Prompt-injection-dataset", "core")
    ds_neuro.save_to_disk(DATA_DIR / "neuralchemy")
    print(f"neuralchemy saved: {len(ds_neuro['train'])} train rows")
else:
    print("neuralchemy: already on disk, skipping.")

if not (DATA_DIR / "spml").exists():
    print("Loading SPML...")
    ds_spml = load_dataset("reshabhs/SPML_Chatbot_Prompt_Injection")
    ds_spml.save_to_disk(DATA_DIR / "spml")
    print(f"SPML saved: {len(ds_spml['train'])} train rows")
else:
    print("SPML: already on disk, skipping.")

print("\nDone.")

## Section 2: EDA from local disk

Loads the snapshots saved by Section 1 and runs exploratory data analysis. This section does not touch HuggingFace and can be re-run freely.


In [4]:
# load from local snapshots from here on
df_deepset = load_from_disk(DATA_DIR / "deepset")["train"].to_pandas()
df_neuro   = load_from_disk(DATA_DIR / "neuralchemy")["train"].to_pandas()
df_spml    = load_from_disk(DATA_DIR / "spml")["train"].to_pandas()

In [5]:
for name, df in [("deepset", df_deepset), ("neuralchemy", df_neuro), ("SPML", df_spml)]:
    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3))


Dataset: deepset
Shape: (546, 2)
Columns: ['text', 'label']
                                                                                                  text  \
0                                                                   Refugee crisis in Europe solutions   
1  My son is very interested in programming and would like to develop his own app. Where can he fin...   
2        I am looking for a new book and would like to know which current bestsellers are recommended.   

   label  
0      0  
1      0  
2      0  

Dataset: neuralchemy
Shape: (4391, 8)
Columns: ['text', 'label', 'category', 'source', 'severity', 'group_id', 'augmented', 'tags']
                                                                   text  \
0              What are the best practices for secure coding in Python?   
1  Can you help me write a Python script that filters input from users?   
2                                  What does 'prompt engineering' mean?   

   label category    source severity

In [6]:
for name, df in [("deepset", df_deepset), ("neuralchemy", df_neuro), ("SPML", df_spml)]:
    print(f"{name}: {list(df.columns)}")

deepset: ['text', 'label']
neuralchemy: ['text', 'label', 'category', 'source', 'severity', 'group_id', 'augmented', 'tags']
SPML: ['System Prompt', 'User Prompt', 'Prompt injection', 'Degree', 'Source']


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

datasets = [
    ("deepset",     df_deepset, "label"),
    ("neuralchemy", df_neuro,   "label"),
    ("SPML",        df_spml,    "Prompt injection"),
]

for ax, (name, df, col) in zip(axes, datasets):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=["steelblue", "coral"])
    ax.set_title(name)
    ax.set_xlabel(f"Label column: '{col}'")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha="center", fontsize=10)

plt.suptitle("Label distributions", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "label_distributions.png", dpi=150)
plt.show()

In [ ]:
# neuralchemy has attack categories - most interesting for your analysis
if "category" in df_neuro.columns:
    print("neuralchemy category breakdown:")
    print(df_neuro["category"].value_counts())
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_neuro["category"].value_counts().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("neuralchemy: attack categories")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "neuralchemy_categories.png", dpi=150)
    plt.show()

In [ ]:
# text length distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

text_cols = [
    ("deepset",     df_deepset, "text"),
    ("neuralchemy", df_neuro,   "text"),
    ("SPML",        df_spml,    "User Prompt"),
]

for ax, (name, df, col) in zip(axes, text_cols):
    lengths = df[col].astype(str).str.len()
    ax.hist(lengths, bins=50, color="steelblue", edgecolor="white")
    ax.set_title(f"{name}: text length")
    ax.set_xlabel("Characters")
    ax.set_ylabel("Count")

plt.suptitle("Prompt length distributions", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "text_lengths.png", dpi=150)
plt.show()

In [12]:
# summary table
summary = []
for name, df, text_col, label_col in [
    ("deepset",     df_deepset, "text",        "label"),
    ("neuralchemy", df_neuro,   "text",        "label"),
    ("SPML",        df_spml,    "User Prompt", "Prompt injection"),
]:
    counts = df[label_col].value_counts()
    total = len(df)
    summary.append({
        "dataset":       name,
        "total_rows":    total,
        "label_column":  label_col,
        "unique_labels": sorted(df[label_col].unique().tolist()),
        "label_counts":  counts.to_dict(),
        "avg_text_len":  round(df[text_col].astype(str).str.len().mean(), 0),
    })

pd.DataFrame(summary)

,dataset,total_rows,label_column,unique_labels,label_counts,avg_text_len
0,deepset,546,label,"[0, 1]","{0: 343, 1: 203}",118.0
1,neuralchemy,4391,label,"[0, 1]","{1: 2650, 0: 1741}",143.0
2,SPML,16012,Prompt injection,"[0, 1]","{1: 12542, 0: 3470}",603.0
